# 12 - Preparación de la Capa de Negocio para Power BI
## Script de Post-Procesamiento: Modelo de Margen Dinámico (ROI)
**Objetivo:** Generar la lógica de negocio financiera desacoplada de los modelos ML.  
El script toma las predicciones agronómicas puras y las transforma en indicadores financieros  
mediante un sistema de reglas heurísticas basado en eficiencia, sostenibilidad y subsidios.

**Outputs generados:**
- `BI_agricola_roi.csv`
- `BI_ganadero_roi.csv`

In [1]:
# =============================================================================
# IMPORTS Y CONFIGURACIÓN
# =============================================================================
import pandas as pd
import numpy as np
import os

# Ruta base de los datos procesados
DATA_PATH = '../data/processed/'

print('Librerías cargadas correctamente.')
print(f'Directorio de datos: {DATA_PATH}')

Librerías cargadas correctamente.
Directorio de datos: ../data/processed/


In [2]:
# =============================================================================
# CARGA DE DATOS
# =============================================================================
df_agricola   = pd.read_csv(os.path.join(DATA_PATH, 'master_agricola.csv'))
df_ganadero   = pd.read_csv(os.path.join(DATA_PATH, 'master_ganadero.csv'))
df_precios    = pd.read_csv(os.path.join(DATA_PATH, 'precios_mercado_processed.csv'))

print(f'master_agricola:          {df_agricola.shape[0]:>7,} filas  x  {df_agricola.shape[1]} columnas')
print(f'master_ganadero:          {df_ganadero.shape[0]:>7,} filas  x  {df_ganadero.shape[1]} columnas')
print(f'precios_mercado_processed:{df_precios.shape[0]:>7,} filas  x  {df_precios.shape[1]} columnas')

master_agricola:            1,200 filas  x  51 columnas
master_ganadero:              600 filas  x  49 columnas
precios_mercado_processed:    864 filas  x  15 columnas


---
## BLOQUE A – Creación del "Oráculo de Precios"
En lugar de usar exclusivamente la media del dataset de precios (que forzaba todos los
cultivos sin precio real a un valor genérico), se definen dos diccionarios con precios
reales de mercado para cada producto/tipo de ganado.

El **Precio Comodín** (media global del dataset de precios) se mantiene como red de seguridad
final para cualquier producto no contemplado en los diccionarios.

In [3]:
# =============================================================================
# BLOQUE A: DICCIONARIOS DE PRECIOS CURADOS
# =============================================================================

# Precio Comodín: media global del dataset de precios (red de seguridad final)
precio_comodin = df_precios['precio_usd_ton'].mean()

# ── Diccionario Agrícola ──────────────────────────────────────────────────────
# Fuentes: precios históricos del dataset (Soja, Arroz, Trigo, Maíz)
#          + estimaciones reales de mercado para cultivos sin cobertura en el dataset
precios_agricola_map = {
    # Productos con precio real del dataset de mercado
    'Soja':           503.80,
    'Arroz':          357.32,
    'Trigo':          303.83,
    'Maíz':           202.28,
    # Productos sin cobertura en el dataset → estimaciones reales de mercado
    'Cebada':         180.00,    # Cereal feed, mercado europeo
    'Caña de azúcar':  35.00,    # Muy alto volumen, precio bajo por tonelada
    'Girasol':        450.00,    # Semilla oleaginosa
    'Algodón':       1800.00,    # Fibra textil
    'Café':          3500.00,    # Cultivo de alto valor
    'Té':            2500.00,    # Cultivo de alto valor
}

# ── Diccionario Ganadero ──────────────────────────────────────────────────────
# Mapeo directo por tipo_ganado → precio USD/ton de carne
precios_ganadero_map = {
    # Productos con precio real del dataset de mercado
    'bovino':  4090.08,
    'porcino': 2519.86,
    # Productos sin cobertura en el dataset → estimaciones reales de mercado
    'avicola': 1600.00,    # Pollo: más barato que porcino
    'ovino':   4500.00,    # Cordero: similar o superior al bovino
    'caprino': 4200.00,    # Cabra
}

print('=== Diccionario Agrícola ===')
for cultivo, precio in precios_agricola_map.items():
    print(f'  {cultivo:<20} {precio:>8.2f} USD/ton')

print(f'\n=== Diccionario Ganadero ===')
for ganado, precio in precios_ganadero_map.items():
    print(f'  {ganado:<20} {precio:>8.2f} USD/ton')

print(f'\n=== Precio Comodín (red de seguridad) ===')
print(f'  Media global dataset precios: {precio_comodin:.2f} USD/ton')

=== Diccionario Agrícola ===
  Soja                   503.80 USD/ton
  Arroz                  357.32 USD/ton
  Trigo                  303.83 USD/ton
  Maíz                   202.28 USD/ton
  Cebada                 180.00 USD/ton
  Caña de azúcar          35.00 USD/ton
  Girasol                450.00 USD/ton
  Algodón               1800.00 USD/ton
  Café                  3500.00 USD/ton
  Té                    2500.00 USD/ton

=== Diccionario Ganadero ===
  bovino                4090.08 USD/ton
  porcino               2519.86 USD/ton
  avicola               1600.00 USD/ton
  ovino                 4500.00 USD/ton
  caprino               4200.00 USD/ton

=== Precio Comodín (red de seguridad) ===
  Media global dataset precios: 1186.33 USD/ton


---
## BLOQUE B – Pipeline Financiero Agrícola
1. **Precio de referencia** → diccionario curado; fallback al Precio Comodín si quedan nulos residuales
2. **Ingreso bruto** → `produccion_ton × precio_referencia_usd`
3. **Margen dinámico** → base 12% ± reglas heurísticas
4. **Beneficio neto** → `ingreso_bruto_usd × margen_pct`

In [5]:
# =============================================================================
# BLOQUE B: PIPELINE FINANCIERO AGRÍCOLA
# =============================================================================

df_bi_agricola = df_agricola.copy()

# ── B1. Precio de Referencia ──────────────────────────────────────────────────
# Prioridad: diccionario curado → Precio Comodín (red de seguridad)
df_bi_agricola['precio_referencia_usd'] = (
    df_bi_agricola['cultivo']
    .map(precios_agricola_map)
    .fillna(precio_comodin)     # seguridad ante cultivos no contemplados
)

# ── B2. Ingreso Bruto ─────────────────────────────────────────────────────────
df_bi_agricola['ingreso_bruto_usd'] = (
    df_bi_agricola['produccion_ton'] * df_bi_agricola['precio_referencia_usd']
)

# ── B3. Margen Dinámico ───────────────────────────────────────────────────────
df_bi_agricola['margen_pct'] = 0.12

umbral_agua = df_bi_agricola['agua_riego_m3_ha'].mean()

# Penalización Hídrica: agua_riego_m3_ha > media global → -0.02
df_bi_agricola['margen_pct'] = np.where(
    df_bi_agricola['agua_riego_m3_ha'] > umbral_agua,
    df_bi_agricola['margen_pct'] - 0.02,
    df_bi_agricola['margen_pct']
)

# Penalización Climática: n_eventos_total_regiones > 0 → -0.03
df_bi_agricola['margen_pct'] = np.where(
    df_bi_agricola['n_eventos_total_regiones'] > 0,
    df_bi_agricola['margen_pct'] - 0.03,
    df_bi_agricola['margen_pct']
)

# Bonificación Estatal: tiene_subsidio_activo == 1 → +0.04
df_bi_agricola['margen_pct'] = np.where(
    df_bi_agricola['tiene_subsidio_activo'] == 1,
    df_bi_agricola['margen_pct'] + 0.04,
    df_bi_agricola['margen_pct']
)

# ── B4. Beneficio Neto ────────────────────────────────────────────────────────
df_bi_agricola['beneficio_neto_usd'] = (
    df_bi_agricola['ingreso_bruto_usd'] * df_bi_agricola['margen_pct']
)

# Diagnóstico de cobertura de precios
cultivos_unicos = df_bi_agricola['cultivo'].unique()
sin_precio_real = [c for c in cultivos_unicos if c not in precios_agricola_map]

print('=== Resumen Pipeline Agrícola ===')
print(f'Umbral agua_riego_m3_ha (media): {umbral_agua:.2f}')
print(f'Cultivos con precio curado:      {len(precios_agricola_map)}')
print(f'Cultivos derivados al Comodín:   {len(sin_precio_real)} → {sin_precio_real if sin_precio_real else "ninguno"}')
print(f'\nEstadísticas margen_pct:')
print(df_bi_agricola['margen_pct'].describe().round(4))
print(f'\nMín: {df_bi_agricola["margen_pct"].min():.4f}  |  Máx: {df_bi_agricola["margen_pct"].max():.4f}')

=== Resumen Pipeline Agrícola ===
Umbral agua_riego_m3_ha (media): 5464.92
Cultivos con precio curado:      10
Cultivos derivados al Comodín:   0 → ninguno

Estadísticas margen_pct:
count    1200.0000
mean        0.1073
std         0.0229
min         0.0700
25%         0.0900
50%         0.1000
75%         0.1200
max         0.1600
Name: margen_pct, dtype: float64

Mín: 0.0700  |  Máx: 0.1600


---
## BLOQUE C – Pipeline Financiero Ganadero
Aplicamos el Modelo de Margen Dinámico sobre `master_ganadero.csv` con reglas específicas de la industria:
1. **Margen base:** 10%
2. **Bonificación por eficiencia:** top 25% de `eficiencia_carne` → +3%
3. **Penalización por contaminación:** `flag_emision_extrema == 1` → -3%
4. **Bonificación estatal:** `tiene_subsidio_activo == 1` → +4%

> **Nota:** El dataset ganadero no tiene columna `produccion_ton` ni precio de cultivo ligado al Oráculo,  
> ya que sus ingresos provienen de `produccion_carne_ton` × precio de carne. Sin embargo, el ticket  
> indica sólo añadir el margen financiero (no hay Bloque B equivalente con precio/ingreso para ganadero),  
> por lo que el pipeline añade exclusivamente `margen_pct` y `beneficio_neto_usd` usando `produccion_carne_ton`  
> y el precio de referencia de la carne más representativa del Oráculo.

1. **Precio de referencia** → diccionario curado por `tipo_ganado`
2. **Ingreso bruto** → `produccion_carne_ton × precio_referencia_usd`
3. **Margen dinámico** → base 10% ± eficiencia, emisiones y subsidio
4. **Beneficio neto** → `ingreso_bruto_usd × margen_pct`

In [7]:
# =============================================================================
# BLOQUE C: PIPELINE FINANCIERO GANADERO
# =============================================================================

df_bi_ganadero = df_ganadero.copy()

# ── C1. Precio de Referencia ──────────────────────────────────────────────────
df_bi_ganadero['precio_referencia_usd'] = (
    df_bi_ganadero['tipo_ganado']
    .map(precios_ganadero_map)
    .fillna(precio_comodin)     # seguridad ante tipos no contemplados
)

# ── C2. Ingreso Bruto ─────────────────────────────────────────────────────────
df_bi_ganadero['ingreso_bruto_usd'] = (
    df_bi_ganadero['produccion_carne_ton'] * df_bi_ganadero['precio_referencia_usd']
)

# ── C3. Margen Dinámico ───────────────────────────────────────────────────────
df_bi_ganadero['margen_pct'] = 0.10

p75_eficiencia = df_bi_ganadero['eficiencia_carne'].quantile(0.75)

# Bonificación por Eficiencia: top 25% de eficiencia_carne → +0.03
df_bi_ganadero['margen_pct'] = np.where(
    df_bi_ganadero['eficiencia_carne'] > p75_eficiencia,
    df_bi_ganadero['margen_pct'] + 0.03,
    df_bi_ganadero['margen_pct']
)

# Penalización por Contaminación: flag_emision_extrema == 1 → -0.03
df_bi_ganadero['margen_pct'] = np.where(
    df_bi_ganadero['flag_emision_extrema'] == 1,
    df_bi_ganadero['margen_pct'] - 0.03,
    df_bi_ganadero['margen_pct']
)

# Bonificación Estatal: tiene_subsidio_activo == 1 → +0.04
df_bi_ganadero['margen_pct'] = np.where(
    df_bi_ganadero['tiene_subsidio_activo'] == 1,
    df_bi_ganadero['margen_pct'] + 0.04,
    df_bi_ganadero['margen_pct']
)

# ── C4. Beneficio Neto ────────────────────────────────────────────────────────
df_bi_ganadero['beneficio_neto_usd'] = (
    df_bi_ganadero['ingreso_bruto_usd'] * df_bi_ganadero['margen_pct']
)

# Diagnóstico de cobertura
ganado_unicos = df_bi_ganadero['tipo_ganado'].unique()
sin_precio_gan = [g for g in ganado_unicos if g not in precios_ganadero_map]

print('=== Resumen Pipeline Ganadero ===')
print(f'Percentil 75 eficiencia_carne:   {p75_eficiencia:.4f}')
print(f'Tipos de ganado con precio curado: {len(precios_ganadero_map)}')
print(f'Tipos derivados al Comodín:        {len(sin_precio_gan)} → {sin_precio_gan if sin_precio_gan else "ninguno"}')
print(f'\nEstadísticas margen_pct:')
print(df_bi_ganadero['margen_pct'].describe().round(4))
print(f'\nMín: {df_bi_ganadero["margen_pct"].min():.4f}  |  Máx: {df_bi_ganadero["margen_pct"].max():.4f}')

=== Resumen Pipeline Ganadero ===
Percentil 75 eficiencia_carne:   0.1334
Tipos de ganado con precio curado: 5
Tipos derivados al Comodín:        0 → ninguno

Estadísticas margen_pct:
count    600.0000
mean       0.1125
std        0.0199
min        0.0700
25%        0.1000
50%        0.1000
75%        0.1300
max        0.1700
Name: margen_pct, dtype: float64

Mín: 0.0700  |  Máx: 0.1700


---
## BLOQUE D – Quality Assurance y Exportación
Verificamos los criterios de aceptación antes de exportar los archivos finales.

In [8]:
# =============================================================================
# BLOQUE D: QUALITY ASSURANCE
# =============================================================================

nuevas_cols_agricola = ['precio_referencia_usd', 'ingreso_bruto_usd', 'margen_pct', 'beneficio_neto_usd']
nuevas_cols_ganadero = ['precio_referencia_usd', 'ingreso_bruto_usd', 'margen_pct', 'beneficio_neto_usd']

# Tolerancia para comparaciones de punto flotante
DECIMALES = 10

print('=' * 60)
print('  QA – BI_agricola_roi')
print('=' * 60)

# QA1: Cero nulos en columnas financieras agrícolas
nulos_agr = df_bi_agricola[nuevas_cols_agricola].isna().sum()
print('\n[CHECK 1] Nulos en columnas financieras agrícolas:')
print(nulos_agr)
assert nulos_agr.sum() == 0, 'ERROR: existen nulos en columnas financieras agrícolas'
print('  ✅ PASS – Cero nulos confirmados')

# QA2: Rango de margen_pct agrícola [0.07, 0.16]
# Se redondea a 10 decimales para evitar falsos negativos por precisión IEEE 754
min_margen_agr = round(df_bi_agricola['margen_pct'].min(), DECIMALES)
max_margen_agr = round(df_bi_agricola['margen_pct'].max(), DECIMALES)
print(f'\n[CHECK 2] Rango margen_pct agrícola: [{min_margen_agr:.4f}, {max_margen_agr:.4f}]')
assert min_margen_agr >= 0.07, f'ERROR: margen mínimo {min_margen_agr:.4f} < 0.07'
assert max_margen_agr <= 0.16, f'ERROR: margen máximo {max_margen_agr:.4f} > 0.16'
print('  ✅ PASS – Rango dentro de [0.07, 0.16]')

# QA3: Columnas originales intactas
n_orig_agr = len(df_agricola.columns)
n_bi_agr   = len(df_bi_agricola.columns)
print(f'\n[CHECK 3] Columnas originales agrícola: {n_orig_agr} → con nuevas: {n_bi_agr} (+ {n_bi_agr - n_orig_agr} nuevas)')
assert n_bi_agr == n_orig_agr + len(nuevas_cols_agricola), 'ERROR: número de columnas no cuadra'
print('  ✅ PASS – Solo se añadieron las columnas financieras al final')

print('\n' + '=' * 60)
print('  QA – BI_ganadero_roi')
print('=' * 60)

# QA4: Cero nulos en columnas financieras ganaderas
nulos_gan = df_bi_ganadero[nuevas_cols_ganadero].isna().sum()
print('\n[CHECK 4] Nulos en columnas financieras ganaderas:')
print(nulos_gan)
assert nulos_gan.sum() == 0, 'ERROR: existen nulos en columnas financieras ganaderas'
print('  ✅ PASS – Cero nulos confirmados')

# QA5: Columnas originales ganaderas intactas
n_orig_gan = len(df_ganadero.columns)
n_bi_gan   = len(df_bi_ganadero.columns)
print(f'\n[CHECK 5] Columnas originales ganadero: {n_orig_gan} → con nuevas: {n_bi_gan} (+ {n_bi_gan - n_orig_gan} nuevas)')
assert n_bi_gan == n_orig_gan + len(nuevas_cols_ganadero), 'ERROR: número de columnas no cuadra'
print('  ✅ PASS – Solo se añadieron las columnas financieras al final')

print('\n✅ TODOS LOS CHECKS DE QA SUPERADOS')

  QA – BI_agricola_roi

[CHECK 1] Nulos en columnas financieras agrícolas:
precio_referencia_usd    0
ingreso_bruto_usd        0
margen_pct               0
beneficio_neto_usd       0
dtype: int64
  ✅ PASS – Cero nulos confirmados

[CHECK 2] Rango margen_pct agrícola: [0.0700, 0.1600]
  ✅ PASS – Rango dentro de [0.07, 0.16]

[CHECK 3] Columnas originales agrícola: 51 → con nuevas: 55 (+ 4 nuevas)
  ✅ PASS – Solo se añadieron las columnas financieras al final

  QA – BI_ganadero_roi

[CHECK 4] Nulos en columnas financieras ganaderas:
precio_referencia_usd    0
ingreso_bruto_usd        0
margen_pct               0
beneficio_neto_usd       0
dtype: int64
  ✅ PASS – Cero nulos confirmados

[CHECK 5] Columnas originales ganadero: 49 → con nuevas: 53 (+ 4 nuevas)
  ✅ PASS – Solo se añadieron las columnas financieras al final

✅ TODOS LOS CHECKS DE QA SUPERADOS


In [9]:
# =============================================================================
# BLOQUE D: EXPORTACIÓN
# =============================================================================

output_dir = '../data/processed/'
os.makedirs(output_dir, exist_ok=True)

path_agricola = os.path.join(output_dir, 'BI_agricola_roi.csv')
path_ganadero = os.path.join(output_dir, 'BI_ganadero_roi.csv')

df_bi_agricola.to_csv(path_agricola, index=False)
df_bi_ganadero.to_csv(path_ganadero, index=False)

print('=== Exportación completada ===')
print(f'  ✅ {path_agricola}')
print(f'     Filas: {len(df_bi_agricola):,}  |  Columnas: {len(df_bi_agricola.columns)}')
print(f'     Nuevas columnas: {nuevas_cols_agricola}')
print()
print(f'  ✅ {path_ganadero}')
print(f'     Filas: {len(df_bi_ganadero):,}  |  Columnas: {len(df_bi_ganadero.columns)}')
print(f'     Nuevas columnas: {nuevas_cols_ganadero}')
print()
print('Los archivos están listos para ser conectados en Power BI.')

=== Exportación completada ===
  ✅ ../data/processed/BI_agricola_roi.csv
     Filas: 1,200  |  Columnas: 55
     Nuevas columnas: ['precio_referencia_usd', 'ingreso_bruto_usd', 'margen_pct', 'beneficio_neto_usd']

  ✅ ../data/processed/BI_ganadero_roi.csv
     Filas: 600  |  Columnas: 53
     Nuevas columnas: ['precio_referencia_usd', 'ingreso_bruto_usd', 'margen_pct', 'beneficio_neto_usd']

Los archivos están listos para ser conectados en Power BI.


---
## Resumen del Modelo de Margen Dinámico

### Sector Agrícola (`BI_agricola_roi.csv`)

| Regla | Condición | Efecto en margen |
|---|---|---|
| Base | Todas las filas | +12% |
| Penalización Hídrica | `agua_riego_m3_ha > media_global` | −2% |
| Penalización Climática | `n_eventos_total_regiones > 0` | −3% |
| Bonificación Estatal | `tiene_subsidio_activo == 1` | +4% |
| **Rango teórico** | | **[7%, 16%]** |

### Sector Ganadero (`BI_ganadero_roi.csv`)

| Regla | Condición | Efecto en margen |
|---|---|---|
| Base | Todas las filas | +10% |
| Bonificación por Eficiencia | `eficiencia_carne > percentil_75` | +3% |
| Penalización Contaminación | `flag_emision_extrema == 1` | −3% |
| Bonificación Estatal | `tiene_subsidio_activo == 1` | +4% |
| **Rango teórico** | | **[7%, 17%]** |